# PCA-based Visualization and K-Means Cluster Discovery (v2)

This notebook implements Step 6: geometric visualization of condition separability in gait feature space using PCA, followed by unsupervised structure discovery with K-Means.

**Data source:** `data/processed/v2/gait_features_v2.csv`  
**Feature set:** v2 feature space from `ALL_FEATURE_COLS` (17 features)  
**Conditions:** PD, HD, ALS (each with disease and control)

Outputs include explained-variance diagnostics, PCA scatter plots, feature-loadings heatmap, K-Means model-selection curves, ARI scores, exported publication figures, and v2 summary tables.

In [1]:
import sys
from pathlib import Path

# Add repository root so `src` is importable in notebook and editor contexts.
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

from src.features import ALL_FEATURE_COLS

matplotlib.rcParams.update({
    'font.size': 9,
    'font.family': 'serif',
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
})

PROCESSED = Path('../data/processed/v2')
RESULTS_V2 = Path('../experiments/results/v2')
FIGURES_PDF = Path('../report/figures/v2/pdf')
FIGURES_PNG = Path('../report/figures/v2/png')
TABLES_V2 = Path('../report/tables/v2')
FIGURES_PDF.mkdir(parents=True, exist_ok=True)
FIGURES_PNG.mkdir(parents=True, exist_ok=True)
TABLES_V2.mkdir(parents=True, exist_ok=True)

COND_ORDER = ['pd', 'hd', 'als', 'control']
COND_LABELS = {'pd': 'PD', 'hd': 'HD', 'als': 'ALS', 'control': 'Healthy control'}
COND_COLORS = {'pd': '#1f77b4', 'hd': '#2ca02c', 'als': '#ff7f0e', 'control': '#7f7f7f'}

print(f'Setup complete. Using {len(ALL_FEATURE_COLS)} features from v2.')

Setup complete. Using 17 features from v2.


In [2]:
features_path = PROCESSED / 'gait_features_v2.csv'
assert features_path.exists(), f'Missing required v2 feature matrix: {features_path}'

gait_df = pl.read_csv(features_path)
print(f'Shape: {gait_df.shape}')
print(gait_df.select(['condition', 'label']).head())

X = gait_df.select(ALL_FEATURE_COLS).to_numpy()
conditions = gait_df['condition'].to_numpy()
labels = gait_df['label'].to_numpy()
subject_ids = gait_df['subject_id'].to_numpy()
cond_label = np.array([f"{c}_{'disease' if y == 1 else 'control'}" for c, y in zip(conditions, labels)])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Condition counts:')
print(pd.Series(conditions).value_counts().sort_index())
print('Disease/control counts:')
print(pd.Series(labels).value_counts().sort_index())
print(f'Feature columns used: {len(ALL_FEATURE_COLS)}')

Shape: (14753, 20)
shape: (5, 2)
┌───────────┬───────┐
│ condition ┆ label │
│ ---       ┆ ---   │
│ str       ┆ i64   │
╞═══════════╪═══════╡
│ als       ┆ 1     │
│ als       ┆ 1     │
│ als       ┆ 1     │
│ als       ┆ 1     │
│ als       ┆ 1     │
└───────────┴───────┘
Condition counts:
als        2404
control    4076
hd         4598
pd         3675
Name: count, dtype: int64
Disease/control counts:
0     4076
1    10677
Name: count, dtype: int64
Feature columns used: 17


In [3]:
pca = PCA(n_components=len(ALL_FEATURE_COLS), random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, ax1 = plt.subplots(figsize=(7, 4))
components = np.arange(1, len(explained_var) + 1)
ax1.bar(components, explained_var, color='#4c72b0', alpha=0.75, label='Per-component variance')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained variance ratio')

ax2 = ax1.twinx()
ax2.plot(components, cumulative_var, color='#dd8452', marker='o', label='Cumulative variance')
ax2.axhline(0.90, color='gray', linestyle='--', linewidth=1)
ax2.axhline(0.95, color='black', linestyle=':', linewidth=1)
ax2.set_ylabel('Cumulative explained variance')
ax2.set_ylim(0.0, 1.02)

ax1.set_title('PCA Explained Variance')
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'pca_explained_variance.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'pca_explained_variance.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'PC1 variance: {explained_var[0]:.4f}')
print(f'PC2 variance: {explained_var[1]:.4f}')
print(f'PC1+PC2 variance: {explained_var[:2].sum():.4f}')

PC1 variance: 0.3750
PC2 variance: 0.2051
PC1+PC2 variance: 0.5801


/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/2179333029.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
pca_plot_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'condition': conditions,
    'condition_label': [COND_LABELS[c] for c in conditions],
    'label': labels,
    'label_name': np.where(labels == 1, 'Disease', 'Control'),
})

fig, ax = plt.subplots(figsize=(7, 5))
for cond in COND_ORDER:
    subset = pca_plot_df[pca_plot_df['condition'] == cond]
    dis = subset[subset['label'] == 1]
    ctl = subset[subset['label'] == 0]
    ax.scatter(
        dis['PC1'], dis['PC2'], s=14, alpha=0.35,
        color=COND_COLORS[cond], marker='o', label=f"{COND_LABELS[cond]} disease"
    )
    ax.scatter(
        ctl['PC1'], ctl['PC2'], s=22, alpha=0.65,
        facecolors='none', edgecolors=COND_COLORS[cond], linewidths=0.8,
        marker='o', label=f"{COND_LABELS[cond]} control"
    )

ax.set_title('PCA Projection: Condition Separability (PC1 vs PC2)')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(alpha=0.2)
ax.legend(loc='best', fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'pca_scatter_by_condition.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'pca_scatter_by_condition.png', dpi=300, bbox_inches='tight')
plt.show()

fig_plotly = px.scatter(
    pca_plot_df,
    x='PC1',
    y='PC2',
    color='condition_label',
    symbol='label_name',
    opacity=0.5,
    title='Interactive PCA Scatter (PC1 vs PC2)'
)
fig_plotly.update_layout(template='plotly_white')
fig_plotly.show()

/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/1073368428.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=ALL_FEATURE_COLS,
    columns=['PC1', 'PC2']
)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(
    loadings,
    cmap='coolwarm',
    center=0.0,
    annot=True,
    fmt='.2f',
    linewidths=0.5,
    cbar_kws={'label': 'Loading weight'},
    ax=ax,
)
ax.set_title('Feature Loadings for PC1 and PC2')
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'pca_feature_loadings.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'pca_feature_loadings.png', dpi=300, bbox_inches='tight')
plt.show()

print('Top absolute loadings in PC1:')
display(loadings['PC1'].abs().sort_values(ascending=False).head(5))
print('Top absolute loadings in PC2:')
display(loadings['PC2'].abs().sort_values(ascending=False).head(5))

Top absolute loadings in PC1:


/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/2326966787.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


double_support_s      0.364638
left_stance_s         0.345054
right_stance_s        0.340785
right_stride_s        0.307195
double_support_pct    0.306024
Name: PC1, dtype: float64

Top absolute loadings in PC2:


right_swing_s       0.462181
left_swing_s        0.408984
left_stride_s       0.333671
right_stance_pct    0.316223
right_swing_pct     0.316223
Name: PC2, dtype: float64

In [6]:
k_values = list(range(2, 11))
inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    preds = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, preds))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(k_values, inertias, marker='o', color='#4c72b0')
ax.set_title('K-Means Elbow Curve')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Inertia')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'kmeans_elbow.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'kmeans_elbow.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(k_values, sil_scores, marker='o', color='#dd8452')
ax.set_title('K-Means Silhouette Scores')
ax.set_xlabel('Number of clusters (K)')
ax.set_ylabel('Silhouette score')
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'kmeans_silhouette.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'kmeans_silhouette.png', dpi=300, bbox_inches='tight')
plt.show()

best_k_by_silhouette = int(k_values[int(np.argmax(sil_scores))])
print(f'Best K by silhouette: {best_k_by_silhouette}')

Best K by silhouette: 2


/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/816856551.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/816856551.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
cond_to_int = {'pd': 0, 'hd': 1, 'als': 2}
disease_mask = np.isin(conditions, list(cond_to_int))
y_condition = np.array([cond_to_int[c] for c in conditions[disease_mask]])
combined_classes = pd.Categorical(cond_label).codes

kmeans3 = KMeans(n_clusters=3, random_state=42, n_init=20)
clusters3 = kmeans3.fit_predict(X_scaled)
ari3 = adjusted_rand_score(y_condition, clusters3[disease_mask])

kmeans6 = KMeans(n_clusters=6, random_state=42, n_init=20)
clusters6 = kmeans6.fit_predict(X_scaled)
ari6 = adjusted_rand_score(combined_classes, clusters6)

print(f'ARI (K=3 vs PD/HD/ALS; pooled healthy-control strides excluded): {ari3:.4f}')
print(f'ARI (K=6 vs condition x label): {ari6:.4f}')

ct3 = pd.crosstab(pd.Series(clusters3, name='cluster_k3'), pd.Series(conditions, name='condition'))
ct6 = pd.crosstab(pd.Series(clusters6, name='cluster_k6'), pd.Series(cond_label, name='condition_label'))

ari_df = pd.DataFrame([
    {'metric': 'ari_k3_vs_condition', 'value': float(ari3)},
    {'metric': 'ari_k6_vs_condition_label', 'value': float(ari6)},
])
ari_df.to_csv(TABLES_V2 / 'pca_kmeans_ari_summary.csv', index=False)
ct3.to_csv(TABLES_V2 / 'pca_kmeans_ct_k3_vs_condition.csv')
ct6.to_csv(TABLES_V2 / 'pca_kmeans_ct_k6_vs_condition_label.csv')

print('\nContingency table (K=3 vs condition):')
display(ct3)
print('\nContingency table (K=6 vs condition x label):')
display(ct6)
print(f'Wrote v2 tables to {TABLES_V2}')

ARI (K=3 vs PD/HD/ALS; pooled healthy-control strides excluded): 0.0338
ARI (K=6 vs condition x label): 0.1678

Contingency table (K=3 vs condition):


condition,als,control,hd,pd
cluster_k3,,,,
0,199,38,1382,1397
1,771,39,441,210
2,1434,3999,2775,2068



Contingency table (K=6 vs condition x label):


condition_label,als_disease,control_control,hd_disease,pd_disease
cluster_k6,,,,
0,4,2,397,46
1,1462,343,936,891
2,214,271,970,1545
3,125,3445,1891,1032
4,596,12,96,153
5,3,3,308,8


Wrote v2 tables to ../report/tables/v2


In [8]:
cluster_plot_df = pca_plot_df.copy()
cluster_plot_df['cluster_k3'] = clusters3.astype(str)

fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    cluster_plot_df['PC1'],
    cluster_plot_df['PC2'],
    c=cluster_plot_df['cluster_k3'].astype(int),
    cmap='tab10',
    s=16,
    alpha=0.5,
)
ax.set_title('K-Means (K=3) Overlay in PCA Space')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(alpha=0.2)
legend = ax.legend(*scatter.legend_elements(), title='Cluster')
ax.add_artist(legend)
fig.tight_layout()
fig.savefig(FIGURES_PDF / 'kmeans_scatter_k3.pdf', bbox_inches='tight')
fig.savefig(FIGURES_PNG / 'kmeans_scatter_k3.png', dpi=300, bbox_inches='tight')
plt.show()

fig_plotly_k3 = px.scatter(
    cluster_plot_df,
    x='PC1',
    y='PC2',
    color='cluster_k3',
    opacity=0.5,
    title='Interactive K-Means (K=3) in PCA Space'
)
fig_plotly_k3.update_layout(template='plotly_white')
fig_plotly_k3.show()

/var/folders/ll/62sh04d569j3dj192d8l9x5r0000gp/T/ipykernel_18598/1127411559.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Clinical Interpretation Prompts

- Which condition appears most separable in the first two principal components?
- Which condition has the strongest overlap with controls?
- Does `K=3` cluster structure align with the three condition labels, or does it reveal a different organization?
- Does `K=6` improve alignment when disease/control status is included?

Use the ARI scores and contingency tables above to ground these interpretations quantitatively.